# 🏥 Sepsis CDSS: Clinical Data Ingestion, EDA & Preprocessing Pipeline
**Project**: Web-Based Clinical Decision Support System (CDSS) for Early Sepsis Risk Detection  
**Clinical Dataset**: PhysioNet / Computing in Cardiology Challenge 2019  

---

### Pipeline Objectives:
1. **Raw Data Ingestion & Concatenation**: Consolidate individual patient `.psv` files or load `physionet_sepsis_combined.csv`.
2. **Exploratory Data Analysis (EDA)**: Profile clinical missingness patterns, demographic breakdowns, and the severe class imbalance (~1.8% positive rate).
3. **Physiological Outlier Cleaning**: Enforce physiological validity ranges (e.g. clipping impossible HR, SBP, or Temp values to NaN).
4. **Patient-Level Longitudinal Imputation**: Perform patient-grouped Forward Fill (`ffill`), Backward Fill (`bfill`), and population median fallback.
5. **Clinical Biomarker & Feature Engineering**:
   - **SIRS Criteria** (Systemic Inflammatory Response Syndrome: Temp, HR, Resp, WBC)
   - **Shock Index** ($HR / SBP$)
   - **qSOFA Score** (quick Sequential Organ Failure Assessment)
   - **3-Hour Rolling Vital Trajectories** ($\Delta HR_{3h}, \Delta MAP_{3h}$)
6. **Patient-Level Stratified Partitioning & Scaling**: Prevent data leakage across records of the same patient and fit a `RobustScaler`.
7. **Dataset Persistence**: Export the cleaned dataset to `physionet_sepsis_processed.csv`.

In [ ]:
import os
import glob
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
import joblib

# Plotting configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline
print("Libraries imported successfully!")

## Step 1: Raw Data Ingestion & Stitching
The raw PhysioNet 2019 challenge data consists of thousands of individual pipe-delimited (`.psv`) files (one file per ICU admission).
If `physionet_sepsis_combined.csv` is already present, we load it directly; otherwise, we stitch all individual `.psv` files from `archive/` into a single combined DataFrame.

In [ ]:
combined_path = 'physionet_sepsis_combined.csv'
if not os.path.exists(combined_path) and os.path.exists('../physionet_sepsis_combined.csv'):
    combined_path = '../physionet_sepsis_combined.csv'

if os.path.exists(combined_path):
    print(f"Loading existing combined dataset from: {combined_path}")
    df = pd.read_csv(combined_path)
else:
    print("Stitching raw PSV files from archive directories...")
    path_a = glob.glob('./archive/training_setA/training_setA/*.psv') or glob.glob('../archive/training_setA/training_setA/*.psv')
    path_b = glob.glob('./archive/training_setB/*.psv') or glob.glob('../archive/training_setB/*.psv')
    all_files = path_a + path_b
    print(f"Found {len(all_files):,} patient files. Concatenating...")
    
    dfs = []
    for f in all_files:
        patient_id = os.path.basename(f).split('.')[0]
        temp_df = pd.read_csv(f, sep='|')
        temp_df['Patient_ID'] = patient_id
        dfs.append(temp_df)
    
    df = pd.concat(dfs, ignore_index=True)
    df.to_csv('physionet_sepsis_combined.csv', index=False)
    print("Combined dataset saved as 'physionet_sepsis_combined.csv'.")

print(f"Total Rows (Hourly ICU Readings): {len(df):,}")
print(f"Unique Patients: {df['Patient_ID'].nunique():,}")
print(f"Total Columns ({len(df.columns)}): {list(df.columns)}")
df.head()

## Step 2: Exploratory Data Analysis (EDA)
### 2.1 Target Class Distribution (`SepsisLabel`)
Sepsis occurs in only a small fraction of ICU patient hours. Let us inspect the exact label distribution and severity of class imbalance.

In [ ]:
sepsis_counts = df['SepsisLabel'].value_counts()
sepsis_pct = df['SepsisLabel'].value_counts(normalize=True) * 100

print("Target Class Breakdown:")
print(f" - Non-Sepsis (0): {sepsis_counts.get(0, 0):,} ({sepsis_pct.get(0, 0):.2f}%)")
print(f" - Sepsis (1):     {sepsis_counts.get(1, 0):,} ({sepsis_pct.get(1, 0):.2f}%)")

fig, ax = plt.subplots(1, 2, figsize=(14, 4))

sepsis_counts.plot(kind='bar', color=['#2b5c8f', '#d9534f'], ax=ax[0])
ax[0].set_title('Hourly Sepsis Label Frequency', fontsize=12, fontweight='bold')
ax[0].set_xticklabels(['Non-Sepsis (0)', 'Sepsis (1)'], rotation=0)
ax[0].set_ylabel('Observation Count')

ax[1].pie(sepsis_counts, labels=['Non-Sepsis (0)', 'Sepsis (1)'],
          autopct='%1.2f%%', colors=['#2b5c8f', '#d9534f'], explode=[0, 0.18], startangle=90)
ax[1].set_title('Sepsis Label Proportion', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

### 2.2 Missing Data Profiling
Clinical EHR records have high rates of missing values. Routine vital signs (Heart Rate, Blood Pressure, Respiration) are continuously measured, while laboratory biomarkers (Lactate, WBC, Creatinine) are measured once or twice daily.

In [ ]:
missing_pct = (df.isnull().mean() * 100).sort_values(ascending=False)

plt.figure(figsize=(16, 6))
missing_pct.plot(kind='bar', color='#3498db', width=0.8)
plt.title('Percentage of Missing Values per Feature in Raw Dataset', fontsize=14, fontweight='bold')
plt.ylabel('Missing Percentage (%)')
plt.xlabel('Features')
plt.axhline(50, color='orange', linestyle='--', label='50% Missing Threshold')
plt.axhline(90, color='red', linestyle='--', label='90% Missing Threshold')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Features with >90% missing values: {list(missing_pct[missing_pct > 90].index)}")

## Step 3: Outlier Cleaning & Range Validation
We apply physiological bounding to reset impossible readings (e.g. sensor disconnects, noise artifacts) to `NaN` prior to imputation. Any columns with 100% missing values (e.g., `EtCO2`) are safely dropped.

In [ ]:
valid_ranges = {
    'HR': (20, 250),
    'O2Sat': (20, 100),
    'Temp': (25, 43),
    'SBP': (30, 270),
    'MAP': (20, 250),
    'DBP': (20, 200),
    'Resp': (4, 65),
    'pH': (6.5, 8.0),
    'Glucose': (10, 1000),
    'WBC': (0.1, 300),
    'Lactate': (0.1, 35)
}

cleaned_outliers = 0
for col, (min_val, max_val) in valid_ranges.items():
    if col in df.columns:
        invalid_mask = (df[col] < min_val) | (df[col] > max_val)
        cnt = invalid_mask.sum()
        if cnt > 0:
            df.loc[invalid_mask, col] = np.nan
            cleaned_outliers += cnt
            print(f" - {col:10s}: reset {cnt:6,d} out-of-range values [{min_val}, {max_val}] to NaN.")

# Drop columns with 100% missing values
all_null_cols = df.columns[df.isnull().mean() == 1.0].tolist()
if all_null_cols:
    print(f"Dropping 100% missing columns: {all_null_cols}")
    df = df.drop(columns=all_null_cols)

print(f"Outlier cleaning complete. Total invalid measurements reset: {cleaned_outliers:,}")

## Step 4: Patient Longitudinal Imputation
To preserve clinical temporal integrity without introducing lookahead bias:
1. **Patient-level Forward Fill (`ffill`)**: Last Observation Carried Forward (LOCF).
2. **Patient-level Backward Fill (`bfill`)**: Initial readings for early ICU hours before labs return.
3. **Population Median Imputation**: Fallback for remaining unobserved measurements across the entire cohort.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'SepsisLabel' in numeric_cols:
    numeric_cols.remove('SepsisLabel')

print("1. Performing Patient-level Forward Fill (ffill)...")
df[numeric_cols] = df.groupby('Patient_ID')[numeric_cols].ffill()

print("2. Performing Patient-level Backward Fill (bfill)...")
df[numeric_cols] = df.groupby('Patient_ID')[numeric_cols].bfill()

print("3. Imputing remaining NaNs with Population Median...")
pop_medians = df[numeric_cols].median()
df[numeric_cols] = df[numeric_cols].fillna(pop_medians)

print(f"Remaining null values across numeric columns: {df[numeric_cols].isnull().sum().sum()}")

## Step 5: Clinical Biomarker & Feature Engineering
We compute established clinical criteria that clinicians use at the bedside:
- **SIRS Criteria**: Evaluates systemic inflammation across Temperature, Heart Rate, Respiration Rate, and WBC.
- **Shock Index**: Calculated as $\text{Shock Index} = \frac{HR}{SBP}$; values $> 0.7$ indicate occult hemodynamic collapse.
- **qSOFA Score**: Quick SOFA score evaluating bedside sepsis morbidity ($Resp \ge 22$, $SBP \le 100$).
- **3-Hour Rolling Vital Deltas**: Captures sudden deterioration trajectories in Heart Rate and MAP.

In [ ]:
# 5.1 SIRS Criteria Calculation
df['SIRS_Temp'] = ((df['Temp'] > 38.0) | (df['Temp'] < 36.0)).astype(int)
df['SIRS_HR'] = (df['HR'] > 90.0).astype(int)
paco2_ref = df['PaCO2'] if 'PaCO2' in df.columns else pd.Series(40.0, index=df.index)
df['SIRS_Resp'] = ((df['Resp'] > 20.0) | (paco2_ref < 32.0)).astype(int)
df['SIRS_WBC'] = ((df['WBC'] > 12.0) | (df['WBC'] < 4.0)).astype(int)
df['SIRS_Score'] = df['SIRS_Temp'] + df['SIRS_HR'] + df['SIRS_Resp'] + df['SIRS_WBC']
df['SIRS_Positive'] = (df['SIRS_Score'] >= 2).astype(int)

# 5.2 Shock Index (HR / SBP)
df['Shock_Index'] = (df['HR'] / (df['SBP'] + 1e-5)).round(4)

# 5.3 qSOFA Score
df['qSOFA_Resp'] = (df['Resp'] >= 22.0).astype(int)
df['qSOFA_SBP'] = (df['SBP'] <= 100.0).astype(int)
df['qSOFA_Score'] = df['qSOFA_Resp'] + df['qSOFA_SBP']

# 5.4 3-Hour Rolling Trajectory Deltas
df['HR_3h_mean'] = df.groupby('Patient_ID')['HR'].transform(lambda x: x.rolling(3, min_periods=1).mean())
df['HR_delta_3h'] = (df['HR'] - df['HR_3h_mean']).round(2)

df['MAP_3h_mean'] = df.groupby('Patient_ID')['MAP'].transform(lambda x: x.rolling(3, min_periods=1).mean())
df['MAP_delta_3h'] = (df['MAP'] - df['MAP_3h_mean']).round(2)

df = df.drop(columns=['HR_3h_mean', 'MAP_3h_mean'])

print("Engineered Clinical Features Summary:")
df[['SIRS_Score', 'SIRS_Positive', 'Shock_Index', 'qSOFA_Score', 'HR_delta_3h', 'MAP_delta_3h']].describe()

## Step 6: Patient-Level Stratified Partitioning & Scaling
To ensure **zero data leakage**, the dataset is partitioned **by unique Patient ID** (rather than individual hourly rows).  
We fit a `RobustScaler` (resistant to clinical outliers) strictly on the training partition and serialize it as `sepsis_scaler.pkl`.

In [ ]:
patient_ids = df['Patient_ID'].unique()
patient_sepsis = df.groupby('Patient_ID')['SepsisLabel'].max()

train_pids, temp_pids = train_test_split(
    patient_ids, test_size=0.30, random_state=42, stratify=patient_sepsis[patient_ids]
)
val_pids, test_pids = train_test_split(
    temp_pids, test_size=0.50, random_state=42, stratify=patient_sepsis[temp_pids]
)

print(f"Total Unique Patients: {len(patient_ids):,}")
print(f" - Train Cohort: {len(train_pids):,} patients ({len(train_pids)/len(patient_ids)*100:.1f}%)")
print(f" - Val Cohort:   {len(val_pids):,} patients ({len(val_pids)/len(patient_ids)*100:.1f}%)")
print(f" - Test Cohort:  {len(test_pids):,} patients ({len(test_pids)/len(patient_ids)*100:.1f}%)")

# Fit RobustScaler on training partition
feature_cols = [c for c in df.columns if c not in ['Patient_ID', 'SepsisLabel']]
scaler = RobustScaler()
train_mask = df['Patient_ID'].isin(train_pids)
scaler.fit(df.loc[train_mask, feature_cols])

# Serialize scaler and metadata
joblib.dump(scaler, 'sepsis_scaler.pkl')
with open('feature_columns.json', 'w') as f:
    json.dump({
        'feature_cols': feature_cols,
        'total_features': len(feature_cols),
        'target_col': 'SepsisLabel',
        'id_col': 'Patient_ID'
    }, f, indent=4)

print("RobustScaler saved to 'sepsis_scaler.pkl' and metadata saved to 'feature_columns.json'.")

## Step 7: Saving the Processed Dataset
We persist the clean dataset to `physionet_sepsis_processed.csv` for model training and research benchmarking.

In [ ]:
output_csv_path = 'physionet_sepsis_processed.csv'
df.to_csv(output_csv_path, index=False)
print(f"SUCCESS: Processed dataset saved to '{output_csv_path}'.")
print(f"Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"File Size: {os.path.getsize(output_csv_path)/(1024*1024):.1f} MB")

---
# Part 2: Model Training, Benchmarking & CDSS Deployment

In [ ]:
import os
import sys
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve
)
import joblib

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline
print("Libraries imported successfully!")

## Step 1: Feature Contract Definition
The CDSS inference engine (`services/ml_service.py`) mandates the **15 Optimal Clinical Features**:
- **7 Vital Signs**: `HR`, `O2Sat`, `Temp`, `SBP`, `MAP`, `DBP`, `Resp`
- **5 Laboratory Biomarkers**: `WBC`, `Glucose`, `Lactate`, `Creatinine`, `Platelets`
- **3 Demographic & Stay Attributes**: `Age`, `Gender`, `ICULOS`

*Clinical Rationale*: PhysioNet features 16-40 suffer from $>90\%$ missingness in real-world ICUs. Training on 15 core biomarkers avoids synthetic imputation noise and delivers superior generalization.

In [ ]:
OPTIMAL_FEATURES = [
    "HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp",
    "WBC", "Glucose", "Lactate", "Creatinine", "Platelets",
    "Age", "Gender", "ICULOS"
]

print(f"15 Optimal Clinical Features ({len(OPTIMAL_FEATURES)}):")
for idx, feat in enumerate(OPTIMAL_FEATURES, start=1):
    print(f"  {idx:2d}. {feat}")

## Step 2: Dataset Loading & Imputation Medians
We load `physionet_sepsis_processed.csv` (or `physionet_sepsis_combined.csv`). We compute population medians for the 15 features and persist them to `model/feature_medians.json` for fallback imputation during live inference.

In [ ]:
csv_paths = [
    'physionet_sepsis_processed.csv',
    '../physionet_sepsis_processed.csv',
    'physionet_sepsis_combined.csv',
    '../physionet_sepsis_combined.csv'
]

dataset_path = None
for path in csv_paths:
    if os.path.exists(path):
        dataset_path = path
        break

if dataset_path is None:
    raise FileNotFoundError("Could not find sepsis dataset CSV file.")

print(f"Loading dataset from: '{dataset_path}'")
df = pd.read_csv(dataset_path)
print(f"Loaded {len(df):,} records with {len(df.columns)} columns.")

X = df[OPTIMAL_FEATURES].copy()
y = df['SepsisLabel'].values

# Compute and serialize feature medians
medians = X.median().to_dict()
os.makedirs('model', exist_ok=True)
with open('model/feature_medians.json', 'w') as f:
    json.dump(medians, f, indent=4)
print("Saved feature medians to 'model/feature_medians.json':")
for k, v in medians.items():
    print(f"  {k:12s}: {v:.2f}")

# Impute any remaining NaNs with column medians
X_imputed = X.fillna(medians)

## Step 3: Stratified Train / Test Split
To evaluate generalization on unseen data while preserving class proportions, we partition the dataset into 80% Training and 20% Testing subsets stratified by `y` (`SepsisLabel`).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Total Records:    {len(X_imputed):,}")
print(f"Training Subset:  {len(X_train):,} (Sepsis: {y_train.sum():,} | {y_train.mean()*100:.2f}%)")
print(f"Testing Subset:   {len(X_test):,} (Sepsis: {y_test.sum():,} | {y_test.mean()*100:.2f}%)")

## Step 4: Multi-Model Training & Benchmarking
We benchmark 3 models representing distinct algorithmic paradigms:
1. **Logistic Regression**: Scaled linear baseline (`class_weight='balanced'`).
2. **Decision Tree**: Fully interpretable hierarchical rule tree (`max_depth=8`, `class_weight='balanced'`).
3. **Random Forest Classifier**: Ensemble of 100 decorrelated trees (`max_depth=15`, `class_weight='balanced'`, `n_jobs=-1`).

In [ ]:
models = {}

# 1. Logistic Regression (requires feature standardization)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training 1/3: Logistic Regression...")
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
t0 = time.time()
lr.fit(X_train_scaled, y_train)
lr_time = time.time() - t0
models['Logistic Regression'] = {'model': lr, 'is_scaled': True, 'train_time': lr_time}

# 2. Single Decision Tree
print("Training 2/3: Decision Tree Classifier...")
dt = DecisionTreeClassifier(max_depth=8, class_weight='balanced', random_state=42)
t0 = time.time()
dt.fit(X_train, y_train)
dt_time = time.time() - t0
models['Decision Tree'] = {'model': dt, 'is_scaled': False, 'train_time': dt_time}

# 3. Random Forest Classifier
print("Training 3/3: Random Forest Classifier (100 Trees)...")
rf = RandomForestClassifier(n_estimators=100, max_depth=15, class_weight='balanced', random_state=42, n_jobs=-1)
t0 = time.time()
rf.fit(X_train, y_train)
rf_time = time.time() - t0
models['Random Forest'] = {'model': rf, 'is_scaled': False, 'train_time': rf_time}

print("All models trained successfully!")

## Step 5: Comprehensive Clinical Performance Evaluation
We evaluate clinical discrimination across all standard diagnostic metrics:
- **Sensitivity / Recall**: Percentage of actual sepsis cases correctly flagged.
- **Specificity**: Percentage of non-septic patients correctly identified (preventing alarm fatigue).
- **PR-AUC (Precision-Recall Area Under Curve)**: The gold standard metric for severe class imbalance.
- **ROC-AUC (Receiver Operating Characteristic Area Under Curve)**: Overall discriminatory power.

In [ ]:
results = []
predictions_dict = {}
probabilities_dict = {}

for name, item in models.items():
    clf = item['model']
    X_eval = X_test_scaled if item['is_scaled'] else X_test
    
    preds = clf.predict(X_eval)
    probs = clf.predict_proba(X_eval)[:, 1]
    
    predictions_dict[name] = preds
    probabilities_dict[name] = probs
    
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds, zero_division=0),
        'Recall / Sensitivity': recall_score(y_test, preds),
        'Specificity': tn / (tn + fp),
        'F1-Score': f1_score(y_test, preds),
        'ROC-AUC': roc_auc_score(y_test, probs),
        'PR-AUC': average_precision_score(y_test, probs),
        'Train Time (s)': item['train_time']
    })

comparison_df = pd.DataFrame(results).set_index('Model')
print("--- CLINICAL PERFORMANCE COMPARISON TABLE ---")
display_df = comparison_df.round(4)
display_df

## Step 6: Diagnostic Visualizations: ROC & PR Curves
- **ROC Curves**: Display True Positive Rate vs False Positive Rate.
- **Precision-Recall Curves**: Critical for medical alerting where False Positives induce clinician alarm fatigue.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = {'Logistic Regression': '#e67e22', 'Decision Tree': '#2ecc71', 'Random Forest': '#2980b9'}

# 1. ROC Curves
for name, probs in probabilities_dict.items():
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc_val = roc_auc_score(y_test, probs)
    axes[0].plot(fpr, tpr, lw=2.2, color=colors[name], label=f"{name} (AUROC = {auc_val:.3f})")

axes[0].plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Guess (AUC = 0.500)')
axes[0].set_title('Receiver Operating Characteristic (ROC) Comparison', fontsize=13, fontweight='bold')
axes[0].set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11)
axes[0].set_ylabel('True Positive Rate (Sensitivity / Recall)', fontsize=11)
axes[0].legend(loc='lower right', frameon=True)

# 2. Precision-Recall Curves
baseline_prec = y_test.mean()
for name, probs in probabilities_dict.items():
    prec, rec, _ = precision_recall_curve(y_test, probs)
    pr_auc_val = average_precision_score(y_test, probs)
    axes[1].plot(rec, prec, lw=2.2, color=colors[name], label=f"{name} (PR-AUC = {pr_auc_val:.3f})")

axes[1].axhline(baseline_prec, color='black', linestyle='--', label=f'Baseline Rate ({baseline_prec:.3f})')
axes[1].set_title('Precision-Recall (PR) Curve Comparison', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Recall (Sensitivity)', fontsize=11)
axes[1].set_ylabel('Precision (Positive Predictive Value)', fontsize=11)
axes[1].legend(loc='upper right', frameon=True)

plt.tight_layout()
plt.show()

## Step 7: Confusion Matrices
Detailed breakdown of True Negatives, False Positives (over-alerting), False Negatives (missed sepsis), and True Positives.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, preds) in zip(axes, predictions_dict.items()):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Pred Non-Sepsis', 'Pred Sepsis'],
                yticklabels=['Actual Non-Sepsis', 'Actual Sepsis'])
    ax.set_title(f'{name} Confusion Matrix', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

## Step 8: Random Forest Feature Importance
Gini Impurity (Mean Decrease in Impurity) reveals the relative contribution of each clinical biomarker to the ensemble's sepsis predictions.

In [ ]:
importances = rf.feature_importances_
feat_imp_df = pd.DataFrame({
    'Feature': OPTIMAL_FEATURES,
    'Importance': importances
}).sort_values(by='Importance', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(feat_imp_df['Feature'], feat_imp_df['Importance'], color='#2b5c8f', edgecolor='navy')
plt.title('Random Forest Feature Importance (15 Optimal Clinical Biomarkers)', fontsize=13, fontweight='bold')
plt.xlabel('Gini Importance (MDI)', fontsize=11)
plt.ylabel('Clinical Features', fontsize=11)
plt.tight_layout()
plt.show()

print("Top Clinical Predictors of Sepsis Risk:")
for _, row in feat_imp_df.tail(6).iloc[::-1].iterrows():
    print(f"  - {row['Feature']:12s}: {row['Importance']:.4f}")

## Step 9: Production Model Export & Startup Verification
We serialize the trained Random Forest model to `sepsis_model.pkl`.  
We then execute the **Startup Contract Verification** to guarantee exact compatibility with `services/ml_service.py` in the Flask CDSS web app.

In [ ]:
# Save model artifact
output_model_path = 'sepsis_model.pkl'
joblib.dump(rf, output_model_path)
print(f"Saved production model to '{output_model_path}' ({os.path.getsize(output_model_path)/(1024*1024):.1f} MB).")

# Verification: Load back and test feature contract
loaded_model = joblib.load(output_model_path)
loaded_features = list(getattr(loaded_model, "feature_names_in_", []))

print(f"Loaded model features ({len(loaded_features)}): {loaded_features}")
if loaded_features == OPTIMAL_FEATURES:
    print("SUCCESS: Model feature names exactly match OPTIMAL_FEATURES contract!")
else:
    print("WARNING: Feature names mismatch!")

# Simulate test inference with population median vitals
test_sample = pd.DataFrame([medians])
prob = loaded_model.predict_proba(test_sample)[0, 1]
print(f"Test inference with median patient vector -> Sepsis Probability: {prob:.4f}")
print("Sepsis CDSS model is fully trained, verified, and ready for deployment!")